# R00 — Audit: do the committed artefacts reproduce the numbers in the paper?

**CPU only, ~1 min, no images needed.** Reads only files that are committed in this repository.

Purpose (reviewer comments R2.2, R2.9 and the internal consistency items of the revision plan):
1. Rebuild every headline number of Table 4 / Table 5 from the committed CSVs and compare it with the manuscript.
2. Test whether the three confirmation artefacts (`*_patchcore_evaluation.csv`, `decomposition_per_candidate_*.csv`,
   `captured_scores/cf_*.npz`) describe the **same** models.
3. Test whether the "Section R" retrieval-baseline confirmations for MVTec were run on MVTec data
   (implied-prevalence check).
4. Quantify the MVTec protocol gap (per-category HPO objective vs pooled confirmation).

Outputs: `review/outputs/tables/R00_*.csv|tex`, `review/outputs/figures/R00_*`, and `review/outputs/frozen_numbers.json`.

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# R0.1) Paper values (typed in from the submitted PDF, Tables 4 and 5)
# ============================================================
PAPER_T4 = pd.DataFrame([
    # dataset, method, distance, time_s, surrogate_f1, pc_f1_mean, pc_f1_sd, gap
    ("mvtec", "ACFX", 0.91, 99.84, 0.779, 0.607, 0.023, -0.172),
    ("mvtec", "CFNOW", 5.65, 5.14, 0.815, 0.569, 0.003, -0.246),
    ("mvtec", "DiCE", 1.58, 0.36, 0.773, 0.623, 0.017, -0.150),
    ("proditec", "ACFX", 0.80, 111.37, 0.802, 0.751, 0.026, -0.051),
    ("proditec", "CFNOW", 3.82, 1.49, 0.897, 0.887, 0.005, -0.011),
    ("proditec", "DiCE", 1.55, 0.34, 0.818, 0.806, 0.047, -0.012),
], columns=["dataset", "method", "distance", "time_s", "surrogate_f1", "pc_f1_mean", "pc_f1_sd", "gap"])
PAPER_T5 = {  # Table 5, means over the 24 confirmed candidates
    "mvtec":    dict(surrogate=0.789, deployed=0.599, oracle=0.832, auroc=0.497),
    "proditec": dict(surrogate=0.839, deployed=0.815, oracle=0.839, auroc=0.947),
}
PAPER_TEXT = dict(rerun_max_abs_diff=0.019, mvtec_val_f1=0.585, mvtec_perclass_auroc=0.827, catwise_gain=0.248,
                  surrogate_dev_r2=0.996, surrogate_grouped_r2=0.995)

In [ ]:
# ============================================================
# R0.2) Load committed artefacts
# ============================================================
D = {}
for ds, s in DS.items():
    cfs = pd.read_csv(s["cfs"]).set_index("cf_id")
    ev = pd.read_csv(s["eval"]).set_index("cf_id")
    dec = pd.read_csv(s["decomp"]).set_index("cf_id")
    mf = pd.read_csv(s["rev"] / f"{ds}_multifactual_candidates.csv")
    npz = {int(p.stem.split("_")[1]): dict(np.load(p)) for p in sorted((s["rev"] / "captured_scores").glob("cf_*.npz"))}
    D[ds] = dict(cfs=cfs, ev=ev, dec=dec, mf=mf, npz=npz)
    print(f"{ds}: candidates={len(cfs)} eval={len(ev)} decomposition={len(dec)} npz={len(npz)} multifactual={len(mf)}")

In [ ]:
# ============================================================
# R0.3) Table 4 rebuilt from the repo vs the manuscript
# ============================================================
rows = []
for ds in DS:
    d = D[ds]
    cfs, ev, dec, mf = d["cfs"], d["ev"], d["dec"], d["mf"]
    for m in ["ACFX", "CFNOW", "DiCE"]:
        c = cfs[cfs.method == m]
        e = ev.loc[c.index]
        dc = dec.loc[c.index]
        mm = mf[mf.method == m]
        p = PAPER_T4[(PAPER_T4.dataset == ds) & (PAPER_T4.method == m)].iloc[0]
        rows.append(dict(
            dataset=ds, method=m,
            paper_distance=p.distance,
            repo_distance_primary=c.distance_to_factual.mean(),
            repo_distance_sweep_mean=mm.distance_to_factual.mean() if len(mm) else np.nan,
            repo_distance_sweep_max=mm.distance_to_factual.max() if len(mm) else np.nan,
            paper_time=p.time_s, repo_time=c.generation_sec.mean(),
            paper_surrogate=p.surrogate_f1, repo_surrogate=c.surrogate_pred.mean(),
            paper_pc_f1=p.pc_f1_mean, repo_pc_f1_evalcsv=e.f1_test.mean(), repo_pc_f1_rerun=dc.f1_test_deployed.mean(),
            paper_pc_sd=p.pc_f1_sd, repo_pc_sd_rerun=dc.f1_test_deployed.std(),
            paper_gap=p.gap, repo_gap_rerun=(dc.f1_test_deployed - dc.surrogate_pred).mean(),
            repo_gap_evalcsv=(e.f1_test - e.surrogate_pred_fresh).mean(),
        ))
T4 = pd.DataFrame(rows)
display(T4.round(3).T)
save_table(T4.round(4), "R00_table4_paper_vs_repo")

In [ ]:
# ============================================================
# R0.4) Table 5 rebuilt, and the sources of each number
# ============================================================
rows = []
for ds in DS:
    d = D[ds]
    dec, ev = d["dec"], d["ev"]
    rows.append(dict(
        dataset=ds,
        paper_surrogate=PAPER_T5[ds]["surrogate"], repo_surrogate_candidates=dec.surrogate_pred.mean(),
        paper_deployed=PAPER_T5[ds]["deployed"], repo_deployed_rerun=dec.f1_test_deployed.mean(),
        repo_deployed_evalcsv=ev.f1_test.mean(),
        paper_oracle=PAPER_T5[ds]["oracle"], repo_oracle_rerun=dec.f1_test_oracle.mean(),
        paper_auroc=PAPER_T5[ds]["auroc"], repo_auroc_rerun=dec.auroc_test.mean(), repo_auroc_evalcsv=ev.auroc_test.mean(),
        repo_val_f1_rerun=dec.f1_val.mean(), repo_val_f1_evalcsv=ev.f1_val_real.mean(),
    ))
T5 = pd.DataFrame(rows)
display(T5.round(3).T)
save_table(T5.round(4), "R00_table5_paper_vs_repo")

In [ ]:
# ============================================================
# R0.5) Are the three confirmation artefacts the SAME models?
#   (a) F1 per candidate: eval-CSV vs decomposition ("rerun")
#   (b) review budget per candidate: candidate file vs captured-scores file
#   (c) implied review budget of the eval-CSV threshold when applied to the captured validation scores
#       (for the same model this must equal the stated budget up to 1/n_val)
# ============================================================
def implied_budget(scores, thr):
    return float(np.mean(scores >= thr))

rows = []
for ds in DS:
    d = D[ds]
    for cf, c in d["cfs"].iterrows():
        e, dc, z = d["ev"].loc[cf], d["dec"].loc[cf], d["npz"][cf]
        b_cfs, b_npz = float(c.params_soft_review_budget), float(z["budget"][0])
        rows.append(dict(
            dataset=ds, cf_id=cf, method=c.method,
            f1_evalcsv=e.f1_test, f1_rerun=dc.f1_test_deployed, abs_diff_f1=abs(e.f1_test - dc.f1_test_deployed),
            budget_candidate_file=b_cfs, budget_captured=b_npz, budget_differs=abs(b_cfs - b_npz) > 1e-6,
            thr_evalcsv=e.threshold_val_budget, val_score_max_captured=float(z["y_score_val"].max()),
            implied_budget_evalthr_on_captured=implied_budget(z["y_score_val"], e.threshold_val_budget),
            stated_budget=b_cfs,
        ))
C = pd.DataFrame(rows)
C["implied_budget_error"] = (C.implied_budget_evalthr_on_captured - C.stated_budget).abs()
C["same_model_plausible"] = (C.implied_budget_error < 0.03) & ~C.budget_differs
display(C.groupby(["dataset", "method"]).agg(
    n=("cf_id", "size"), mean_abs_f1_diff=("abs_diff_f1", "mean"), max_abs_f1_diff=("abs_diff_f1", "max"),
    n_diff_gt_002=("abs_diff_f1", lambda s: int((s > 0.02).sum())), n_budget_differs=("budget_differs", "sum"),
    n_same_model_plausible=("same_model_plausible", "sum")).round(3))
summ = C.groupby("dataset").agg(max_abs_f1_diff=("abs_diff_f1", "max"), mean_abs_f1_diff=("abs_diff_f1", "mean"),
                                n_diff_gt_002=("abs_diff_f1", lambda s: int((s > 0.02).sum())),
                                n_budget_differs=("budget_differs", "sum"),
                                n_thr_above_all_val_scores=("thr_evalcsv", lambda s: 0)).reset_index()
summ["n_thr_above_all_val_scores"] = [int((C[C.dataset == k].thr_evalcsv > C[C.dataset == k].val_score_max_captured).sum())
                                      for k in summ.dataset]
print(f"Paper claims: max rerun difference = {PAPER_TEXT['rerun_max_abs_diff']}")
display(summ.round(3))
save_table(C.round(5), "R00_confirmation_artefact_consistency_per_candidate")
save_table(summ.round(4), "R00_confirmation_artefact_consistency_summary")

**How to read the table above.**
* `abs_diff_f1` is the per-candidate difference between the confirmation CSV and the "rerun" used for Table 4/5.
  The manuscript states the maximum is 0.019.
* `budget_differs` is true if the review budget stored with the captured scores differs from the budget in the committed
  candidate file. For a genuine rerun of the *same* candidate it must be false.
* `implied_budget_evalthr_on_captured` applies the confirmation-CSV threshold to the captured validation scores. For the same
  model it must be ≈ the stated budget. Large errors, or thresholds above every captured validation score, mean the two
  files were produced by different models.
* **Observed pattern.** CFNOW candidates (retrieved rows of the HPO log, deterministic) reproduce across all three artefacts.
  DiCE/ACFX candidates (stochastic, continuous configurations) do **not**: the review budget stored with the captured
  scores differs from the committed candidate file, and the confirmation-CSV threshold is incompatible with the
  captured validation scores (on MVTec it lies above *every* captured validation score for 16/24 candidates).
  The most economical explanation is not seed noise but that the artefacts were produced from different generations
  of the stochastic candidate table (a stale `cf_id → configuration` mapping). This cannot be settled from the files
  alone: R02 re-runs one **frozen** candidate table under a documented protocol and replaces both sets of numbers.
  Until then, Table 4/5 should not be mixed with the confirmation CSV.

In [ ]:
# ============================================================
# R0.6) "Section R" retrieval-baseline confirmations: implied test prevalence
#   precision = TP / predicted_positive ; recall = TP / positives ; predicted_positive ≈ budget * n
#   => prevalence ≈ precision * budget / recall
# ============================================================
prev_true = {"mvtec": 167 / 238, "proditec": 299 / 867}
rows = []
for ds, s in DS.items():
    b = pd.read_csv(s["confirm"])
    b["implied_prevalence"] = b.precision_test * b.params_soft_review_budget / b.recall_test
    b["true_test_prevalence"] = prev_true[ds]
    b["dataset"] = ds
    rows.append(b[["dataset", "qid", "anchor", "method", "params_soft_review_budget", "precision_test", "recall_test",
                   "f1_test", "auroc_test", "wall_s", "implied_prevalence", "true_test_prevalence"]])
S = pd.concat(rows, ignore_index=True)
S["prevalence_consistent"] = (S.implied_prevalence - S.true_test_prevalence).abs() < 0.08
display(S.round(3))
print(S.groupby("dataset").agg(n=("method", "size"), implied_prev=("implied_prevalence", "mean"),
                               true_prev=("true_test_prevalence", "first"), consistent=("prevalence_consistent", "sum"),
                               mean_auroc=("auroc_test", "mean"), mean_wall_s=("wall_s", "mean")).round(3))
save_table(S.round(4), "R00_sectionR_implied_prevalence")

If the MVTec rows of Section R show an implied prevalence ≈ 0.34 (the Proditec value) instead of 0.70, and AUROC ≈ 0.98
with 6–20× the usual wall-time, they were **not** run on MVTec test data. In that case the manuscript sentence that
upper-quantile MVTec candidates retrain to AUROC ≈ 0.98 must not be used; R03 re-runs them correctly.

In [ ]:
# ============================================================
# R0.7) MVTec protocol gap: what the HPO objective measured vs what the confirmation measured
# ============================================================
bp = pd.read_csv(REPO / "mvtec" / "outputs_patchcore" / "best_params_final_results.csv")
bj = json.loads((REPO / "mvtec" / "outputs_patchcore" / "best_params.json").read_text())
prev_pool = 167 / 238
budget = bj["soft_review_budget"]
gap = pd.DataFrame([
    dict(quantity="HPO objective = mean over 4 categories of val F1 (per-category model + threshold)", value="see log: mean_f1_val_at_budget"),
    dict(quantity="best config, per-category FINAL F1 (HPO protocol)", value=bp.f1_final_at_budget.mean()),
    dict(quantity="pooled test prevalence (confirmation protocol)", value=prev_pool),
    dict(quantity="F1 of a PERFECT ranker at the best budget on the pooled set", value=2 * min(budget, prev_pool) / (budget + prev_pool)),
    dict(quantity="F1 of a RANDOM ranker at the best budget on the pooled set", value=2 * prev_pool * budget / (prev_pool + budget)),
    dict(quantity="Proditec test prevalence", value=prev_true["proditec"]),
    dict(quantity="F1 of a RANDOM ranker at budget 0.345 on Proditec", value=2 * prev_true["proditec"] * 0.345 / (prev_true["proditec"] + 0.345)),
])
display(gap)
save_table(gap, "R00_mvtec_protocol_gap")

# review-budget effect on the logged F1 landscape (why the surrogate R² is so high)
log = pd.read_csv(DS["mvtec"]["dataset"])
perfect = 2 * np.minimum(log.params_soft_review_budget, prev_pool) / (log.params_soft_review_budget + prev_pool)
rand = 2 * prev_pool * log.params_soft_review_budget / (prev_pool + log.params_soft_review_budget)
r2_budget_only = 1 - ((log.value - (log.value.groupby(log.params_soft_review_budget).transform("mean"))) ** 2).sum() / ((log.value - log.value.mean()) ** 2).sum()
print(f"MVTec log: share of F1 variance explained by the review budget alone (group means) = {r2_budget_only:.3f}")
print(f"corr(F1, perfect-ranker F1(budget)) = {np.corrcoef(log.value, perfect)[0, 1]:.3f}")

fig, ax = plt.subplots(figsize=SINGLE_COL)
ax.scatter(log.params_soft_review_budget, log.value, s=2, alpha=0.25, color="#3b82d4", label="logged trials")
bb = np.linspace(0.005, 0.5, 100)
ax.plot(bb, 2 * np.minimum(bb, prev_pool) / (bb + prev_pool), color="k", lw=1, label="perfect ranker, prev. 0.70")
ax.plot(bb, 2 * prev_pool * bb / (prev_pool + bb), color="#7c5cd8", lw=1, ls="--", label="random ranker")
ax.set_xlabel("review budget"); ax.set_ylabel("logged val. F1 (MVTec)"); ax.legend(frameon=False)
save_fig(fig, "R00_mvtec_f1_vs_review_budget")

In [ ]:
# ============================================================
# R0.8) Candidate-file consistency: are all 24 MVTec candidates basically the same corner of the space?
# ============================================================
cf = D["mvtec"]["cfs"]
print(cf.groupby("method")[["params_soft_review_budget", "params_soft_train_fraction"]].agg(["min", "max"]).round(3))
print(cf.params_backbone.value_counts().to_dict(), cf.params_layers_key.value_counts().to_dict())
ev = D["mvtec"]["ev"]
print("distinct F1 values in the MVTec eval CSV:", ev.f1_test.round(4).nunique(), "| most common:", ev.f1_test.round(4).value_counts().head(3).to_dict())

In [ ]:
# ============================================================
# R0.9) Freeze
# ============================================================
nums = dict(
    table4_paper_vs_repo=T4.round(4).to_dict(orient="records"),
    table5_paper_vs_repo=T5.round(4).to_dict(orient="records"),
    rerun_vs_evalcsv=summ.round(4).to_dict(orient="records"),
    sectionR_prevalence=S.groupby("dataset").agg(implied=("implied_prevalence", "mean"), true=("true_test_prevalence", "first"),
                                                 auroc=("auroc_test", "mean")).round(4).reset_index().to_dict(orient="records"),
    mvtec_budget_only_variance_share=float(r2_budget_only),
    mvtec_perfect_ranker_f1=float(2 * min(budget, prev_pool) / (budget + prev_pool)),
    mvtec_best_perclass_final_f1=float(bp.f1_final_at_budget.mean()),
)
merge_numbers("R00_audit", nums)